# QQQI / QQQ / partial-TQQQ with VIX

Research-only notebook for the frozen `qqqi_qqq_tqqq_vix_v2` contract. VIX is used as a risk-state signal, never as a tradable spot asset. All close-derived decisions execute at the next open.

In [ ]:
from pathlib import Path

import pandas as pd
import yaml

from src.research.etf_rotation_experiment import fetch_adjusted_daily_bars
from src.research.vix_rotation_experiment import (
    VIX_SYMBOL,
    config_from_contract,
    vix_regime_asset_metrics,
    vix_repair_event_study,
    vix_signal_audit,
)
from src.research.vix_rotation_runtime import run_vix_runtime_comparison

In [ ]:
contract_path = Path('../configs/research_paradigms/qqqi_qqq_tqqq_vix_v2.yaml')
if not contract_path.exists():
    contract_path = Path('configs/research_paradigms/qqqi_qqq_tqqq_vix_v2.yaml')
contract = yaml.safe_load(contract_path.read_text(encoding='utf-8'))
config = config_from_contract(contract)
config

In [ ]:
symbols = [*contract['boundaries']['tradable_symbols'], VIX_SYMBOL]
bars, coverage = fetch_adjusted_daily_bars(
    symbols=symbols,
    start=contract['data']['start_date'],
    end=contract['data'].get('end_date'),
)
coverage

In [ ]:
metrics, results, prepared = run_vix_runtime_comparison(bars, config)
columns = [
    'total_return', 'cagr', 'annual_volatility', 'sharpe',
    'sortino', 'max_drawdown', 'calmar', 'switch_count',
]
metrics[[column for column in columns if column in metrics.columns]]

## Did VIX regimes distinguish QQQI from QQQ?

Regime labels are known at close `t`; measured returns begin at the next executable open.

In [ ]:
vix_regime_asset_metrics(prepared)

## Stress, easing and normalization events

Each event family contributes at most one first trigger per VIX stress cluster.

In [ ]:
events = vix_repair_event_study(
    prepared,
    horizons=contract['validation']['event_horizons'],
    cluster_gap_sessions=contract['validation']['vix_event_cluster_gap_sessions'],
)
events

## State reachability and executed weights

In [ ]:
audit = vix_signal_audit(prepared)
audit

In [ ]:
v2 = results['rotation_vix_v2']
v2.trades

In [ ]:
weight_columns = ['weight_QQQI', 'weight_QQQ', 'weight_TQQQ']
v2.daily[weight_columns].mean().rename('average_weight').to_frame()

## Interpretation guardrail

Do not tune the VIX quantiles or retreat thresholds in this notebook after viewing returns. A revised rule requires a new versioned contract and a separately identified validation window.